# Funding carry robustness

The proof-protocol close. The two winners, smoothed persistence and gbm against raw persistence, on fixed threshold and slow partial adjustment, run at zero, real and two times cost, plus a maker upper bound and a deflated Sharpe. Decides whether the edge survives pessimistic cost and whether the maker engine build is worth starting.

Runs on the cost-aware build, `QP_BACKTEST_MATCHER=cost_aware`, so the passed cost table actually loads. Earlier notebooks ran on a stale binary that undercharged, so their absolute numbers are light, the relative verdicts hold and are reconfirmed here at the correct, higher cost.

## Setup and predictions

Same pipeline, expanding monthly policy, plus smoothed persistence, an ewma of the funding input over halflife 6.

In [1]:
import os, sys, glob, warnings, time, tempfile
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from scipy.stats import norm, skew, kurtosis
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / "research").is_dir():
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import research.features as F
import research.cv as CV
from research.models import _with_symbol_dummies

_so = glob.glob(str(REPO / "build" / "release" / "**" / "qp_python_backtest*.so"), recursive=True)
sys.path.insert(0, os.path.dirname(_so[0]))
import qp_python_backtest as qb

DATA       = REPO / "data" / "binance_historical"
REAL_COST  = str(REPO / "data" / "cost_model" / "cost_table.csv")
SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT",
           "DOGEUSDT", "ADAUSDT", "LINKUSDT", "AVAXUSDT", "LTCUSDT"]
NAME_TO_ID = {n: i for i, n in enumerate(SYMBOLS)}
FUT, SPOT = 0, 1
HORIZON = 24
HURDLE = 0.0034
SOURCES = ["persistence", "psm6", "gbm"]

In [2]:
FUNDING_COLS = ["symbol", "tag", "ts_ms", "interval_h", "funding_rate"]
PREMIUM_COLS = ["ts_ms", "open", "high", "low", "close", "volume", "close_ms",
                "quote_vol", "trades", "taker_buy_base", "taker_buy_quote", "ignore"]


def load_funding(sym):
    d = DATA / sym / "futures" / "funding"
    files = sorted(d.glob(f"{sym}-fundingRate-*.csv"))
    df = pd.concat([pd.read_csv(f, header=None, names=FUNDING_COLS) for f in files], ignore_index=True)
    df["ts"] = pd.to_datetime(df["ts_ms"], unit="ms", utc=True)
    df["symbol"] = sym
    return (df[["symbol", "ts", "funding_rate"]].rename(columns={"funding_rate": "realized_funding"})
            .sort_values("ts").drop_duplicates("ts").reset_index(drop=True))


def load_premium(sym):
    d = DATA / sym / "futures" / "premiumindex"
    files = sorted(d.glob("*.csv"))
    if not files:
        return pd.DataFrame(columns=["ts_ms", "close"])
    parts = [pd.read_csv(f, header=None, names=PREMIUM_COLS)[["ts_ms", "close"]] for f in files]
    df = pd.concat(parts, ignore_index=True).sort_values("ts_ms").drop_duplicates("ts_ms").reset_index(drop=True)
    df["ts"] = pd.to_datetime(df["ts_ms"], unit="ms", utc=True)
    return df


def average_premium_up_to(premium, funding_ts, interval_hours=8):
    if premium.empty:
        return pd.Series(np.nan, index=range(len(funding_ts)))
    p = premium.sort_values("ts").reset_index(drop=True)
    win = pd.Timedelta(hours=interval_hours)
    out = np.full(len(funding_ts), np.nan)
    ts_vals = pd.to_datetime(funding_ts, utc=True).to_numpy()
    p_ts = p["ts"].to_numpy(); p_close = p["close"].to_numpy(dtype=float)
    for i, end in enumerate(ts_vals):
        lo = np.searchsorted(p_ts, end - win, side="right")
        hi = np.searchsorted(p_ts, end, side="right")
        if hi > lo:
            out[i] = p_close[lo:hi].mean()
    return pd.Series(out)


parts = []
for sym in SYMBOLS:
    f = load_funding(sym); p = load_premium(sym)
    f["premium"] = average_premium_up_to(p, f["ts"]).to_numpy()
    parts.append(f)
events = pd.concat(parts, ignore_index=True)

feat = events.copy()
for k in (1, 2, 3):
    feat = F.add_funding_lag(feat, k)
feat = F.add_funding_ewma(feat, halflife=2); feat = F.add_funding_ewma(feat, halflife=6)
feat = F.add_funding_vol(feat, window=8); feat = F.add_clamp_distance(feat)
feat = F.add_premium_trend(feat, span=3); feat = F.add_cross_symbol_spread(feat, reference="BTCUSDT")
feat = F.add_basket_spread(feat); feat = F.add_time_features(feat)
feat = F.add_funding_mean_window(feat, window=90); feat = F.add_funding_sign_window(feat, window=90)
feat = F.add_funding_vol_rank(feat, vol_window=30, rank_window=180)
kline_agg = pd.read_pickle(REPO / "research" / "results" / "kline_aggregates_8h.pkl")
feat = feat.merge(kline_agg, on=["symbol", "ts"], how="left")
feat["taker_imb_x_ewma2"] = feat["taker_imbalance"] * feat["funding_ewma_h2"]
feat["taker_imb_x_lag1"] = feat["taker_imbalance"] * feat["funding_lag1"]
feat["vol1m_x_clamp"] = feat["realized_vol_1m"] * feat["clamp_distance"]
feat["range_x_ewma2"] = feat["high_low_range"] * feat["funding_ewma_h2"]
feat["regime_x_lag1"] = feat["funding_sign_w90"] * feat["funding_lag1"]
feat["basket_x_lag1"] = feat["basket_spread"] * feat["funding_lag1"]
feat["lag1_sq"] = feat["funding_lag1"] ** 2
feat = F.add_basket_zscore(feat, source="funding_lag1")
feat = F.add_basket_zscore(feat, source="funding_ewma_h6")
feat = F.add_basket_rank(feat, source="funding_lag1")
feat["basket_z_x_lag1"] = feat["basket_z_funding_lag1"] * feat["funding_lag1"]
feat["basket_rank_x_lag1"] = feat["basket_rank_funding_lag1"] * feat["funding_lag1"]
feat = F.add_cum_target(feat, horizon=HORIZON)
FCOLS = [c for c in feat.columns
         if c not in ("symbol", "ts", "realized_funding", "premium", "realized_cum")]
feat = feat.dropna(subset=FCOLS + ["realized_funding", "realized_cum"]).reset_index(drop=True)
feat["year"] = feat["ts"].dt.year
folds = list(CV.walk_forward_splits(feat, n_folds=5, horizon=HORIZON, embargo=5))
fold_of = np.full(len(feat), -1)
for fi, (_, te) in enumerate(folds):
    fold_of[te] = fi
feat["fold"] = fold_of

PURGE = pd.Timedelta(hours=HORIZON * 8)


def elastic_pred(train, test, cols):
    Xtr, _ = _with_symbol_dummies(train, cols)
    pipe = Pipeline([("sc", StandardScaler()),
                     ("m", ElasticNet(alpha=1e-4, l1_ratio=0.7, max_iter=20000))])
    pipe.fit(Xtr, train["realized_cum"].to_numpy())
    Xte, _ = _with_symbol_dummies(test, cols)
    return pipe.predict(Xte)


def gbm_pred(train, test, cols):
    import lightgbm as lgb
    Xtr, c = _with_symbol_dummies(train, cols)
    y = train["realized_cum"].to_numpy(); sp = int(len(y) * 0.8)
    p = dict(objective="regression", verbose=-1, feature_fraction=0.9, bagging_fraction=0.9,
             bagging_freq=5, learning_rate=0.02, num_leaves=15, min_data_in_leaf=500)
    dtr = lgb.Dataset(Xtr[:sp], label=y[:sp], feature_name=c)
    dval = lgb.Dataset(Xtr[sp:], label=y[sp:], reference=dtr)
    b = lgb.train(p, dtr, num_boost_round=3000, valid_sets=[dval],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
    Xte, _ = _with_symbol_dummies(test, cols)
    return b.predict(Xte, num_iteration=b.best_iteration)


MODELS = {"elastic": elastic_pred, "gbm": gbm_pred}


def gen_policy_oof(window_months, refit_freq, min_train=2000):
    test = feat[feat.fold >= 0]
    start = pd.Timestamp(test.ts.min()).tz_convert("UTC").normalize().replace(day=1)
    dates = pd.date_range(start, test.ts.max(), freq=refit_freq, tz="UTC")
    di = np.searchsorted(dates.to_numpy(), test.ts.to_numpy(), side="right") - 1
    out = test[["symbol", "ts", "fold", "year", "realized_funding",
                "realized_cum", "funding_lag1"]].copy()
    for name in MODELS:
        out["pred_" + name] = np.nan
    out["pred_persistence"] = (HORIZON * test["funding_lag1"]).to_numpy()
    for k, d in enumerate(dates):
        sl = test[di == k]
        if sl.empty:
            continue
        tr = feat[feat.ts <= d - PURGE]
        if window_months is not None:
            tr = tr[tr.ts >= d - pd.DateOffset(months=window_months)]
        if len(tr) < min_train:
            continue
        for name, fn in MODELS.items():
            out.loc[sl.index, "pred_" + name] = fn(tr, sl, FCOLS)
    return out


oof = gen_policy_oof(None, "MS").dropna(subset=["pred_elastic", "pred_gbm"])
oof = oof.sort_values(["symbol", "ts"]).reset_index(drop=True)
oof["pred_psm6"] = HORIZON * oof.groupby("symbol").funding_lag1.transform(
    lambda s: s.ewm(halflife=6).mean())
print("oof rows", len(oof))

oof rows 22448


## Cost tables

Four cost points from the one real table. Zero nulls every cost column, the cost-free ceiling. Two times scales them, the pessimism check. Maker is the optimistic execution ceiling, no spread cross and no impact because a resting order provides liquidity, fees at the maker rate, 2 bp perp and 7.5 bp spot. It overstates maker because it assumes fills and ignores adverse selection, so it bounds the upside.

In [3]:
_tmp = Path(tempfile.mkdtemp())
_ct = pd.read_csv(REAL_COST)
_cols = ["half_spread_bps", "impact_bps_per_unit", "taker_fee_bps"]


def _write(df, name):
    p = str(_tmp / name); df.to_csv(p, index=False); return p


zero = _ct.copy(); zero[_cols] = 0.0
x2 = _ct.copy(); x2[_cols] *= 2
mk = _ct.copy(); mk["half_spread_bps"] = 0.0; mk["impact_bps_per_unit"] = 0.0
mk.loc[mk.venue == FUT, "taker_fee_bps"] = 2.0
mk.loc[mk.venue == SPOT, "taker_fee_bps"] = 7.5
COSTS = {"zero": _write(zero, "zero.csv"), "real": REAL_COST,
         "2x": _write(x2, "x2.csv"), "maker": _write(mk, "maker.csv")}

In [4]:
class PredThreshold:
    def __init__(self, bt, pred, qty=1.0, enter=0.0034, exit=0.0):
        self.bt = bt; self.pred = pred; self.qty = qty; self.enter = enter; self.exit = exit
    def on_event(self, ev):
        if ev.kind != qb.EventKind.Funding or ev.venue != FUT:
            return None
        p = self.pred.get((ev.symbol, ev.ts))
        if p is None:
            return None
        held = self.bt.position(ev.symbol, SPOT)
        t = self.qty if p >= self.enter else (0.0 if p <= self.exit else held)
        return [qb.Intent(ev.symbol, SPOT, t), qb.Intent(ev.symbol, FUT, -t)]
    def on_timer(self, now):
        return None


class PredPartialAdjust:
    def __init__(self, bt, pred, hurdle=0.0034, cap=2.0, adjust=0.15):
        self.bt = bt; self.pred = pred; self.hurdle = hurdle; self.cap = cap; self.adjust = adjust
    def on_event(self, ev):
        if ev.kind != qb.EventKind.Funding or ev.venue != FUT:
            return None
        p = self.pred.get((ev.symbol, ev.ts))
        if p is None:
            return None
        aim = float(np.clip(p / self.hurdle, 0.0, self.cap))
        cur = self.bt.position(ev.symbol, SPOT)
        t = cur + self.adjust * (aim - cur)
        return [qb.Intent(ev.symbol, SPOT, t), qb.Intent(ev.symbol, FUT, -t)]
    def on_timer(self, now):
        return None


class Gate:
    def __init__(self, bt, cap=10.0):
        self.bt = bt; self.cap = cap; self._id = 1
    def _next(self):
        i = self._id; self._id += 1; return i
    def check(self, it):
        cur = self.bt.position(it.symbol, it.venue)
        t = float(np.clip(it.target_position, -self.cap, self.cap)); d = t - cur
        out = qb.RiskOutcome.Approved if t == it.target_position else qb.RiskOutcome.Resized
        side = qb.Side.Buy if d >= 0 else qb.Side.Sell
        return qb.RiskDecision(out, qb.Order(self._next(), it.symbol, side, it.venue, abs(d)))


PF = oof.ts.min().date().isoformat(); PL = oof.ts.max().date().isoformat()


def pred_map(col):
    d = oof.dropna(subset=[col])
    sid = d.symbol.map(NAME_TO_ID).to_numpy()
    tns = (d.ts.dt.tz_convert("UTC").dt.tz_localize(None)
           .values.astype("datetime64[ns]").astype("int64"))
    return dict(zip(zip(sid, tns), d[col]))


def daily_sharpe(eq):
    dd = eq.set_index("ts")["equity"].resample("1D").last().ffill().diff().dropna()
    return (dd.mean() / dd.std()) * np.sqrt(365) if dd.std() else 0.0


def run(col, mk, cost):
    ds = qb.Dataset(str(DATA), SYMBOLS, PF, PL, cost)
    bt = qb.PythonBacktest(ds)
    st = mk(bt, pred_map(col)); g = Gate(bt)
    bt.set_on_event(st.on_event); bt.set_on_timer(st.on_timer); bt.set_check(g.check)
    bt.set_event_kinds([qb.EventKind.Funding])
    r = bt.run()
    eq = pd.DataFrame({"ts": pd.to_datetime([p.ts for p in r.equity_series], utc=True),
                       "equity": [p.equity for p in r.equity_series]})
    return r.final_equity, daily_sharpe(eq), eq


STRATS = {"threshold": lambda bt, p: PredThreshold(bt, p, 1.0, 0.0034, 0.0),
          "partial": lambda bt, p: PredPartialAdjust(bt, p, 0.0034, 2.0, 0.15)}
rows = []
eqs = {}
for sname, mkf in STRATS.items():
    for src in SOURCES:
        for cn, cp in COSTS.items():
            fe, sh, eq = run("pred_" + src, mkf, cp)
            rows.append({"strategy": sname, "source": src, "cost": cn,
                         "final": round(fe, 1), "sharpe": round(sh, 2)})
            if cn == "real":
                eqs[(sname, src)] = eq
res = pd.DataFrame(rows)
for sname in STRATS:
    print(f"\n=== {sname} final PnL ===")
    display(res[res.strategy == sname].pivot(index="source", columns="cost",
            values="final")[["zero", "real", "2x", "maker"]])
    print(f"=== {sname} Sharpe ===")
    display(res[res.strategy == sname].pivot(index="source", columns="cost",
            values="sharpe")[["zero", "real", "2x", "maker"]])


=== threshold final PnL ===


cost,zero,real,2x,maker
source,,,,
gbm,7998.5,7638.6,7278.6,7814.3
persistence,8960.1,8133.3,7306.4,8508.7
psm6,9036.4,8674.9,8313.4,8853.4


=== threshold Sharpe ===


cost,zero,real,2x,maker
source,,,,
gbm,4.27,4.00,3.64,4.15
persistence,5.24,4.58,3.80,4.91
psm6,4.75,4.46,4.09,4.62



=== partial final PnL ===


cost,zero,real,2x,maker
source,,,,
gbm,11538.8,9586.8,7634.8,10231.5
persistence,12106.1,7778.9,3451.8,9224.5
psm6,12190.2,10339.6,8489.0,10947.7


=== partial Sharpe ===


cost,zero,real,2x,maker
source,,,,
gbm,4.83,4.08,3.28,4.33
persistence,4.73,3.08,1.37,3.64
psm6,4.65,3.99,3.30,4.21


Smoothed persistence wins at every cost on both rules and survives two times cost, partial 8489 at 2x against raw persistence's 3451. Raw persistence partial collapses under cost, its turnover is the fragility, which is exactly what the ewma fixes. gbm stays dominated by smoothed persistence everywhere. The edge is robust, the sign survives two times cost with a Sharpe above 3.

The maker ceiling is thin for the winner, smoothed persistence partial goes 10340 to 10948, about 6 percent, and that is the optimistic bound that assumes fills. The low-turnover winner pays little in fees, so maker has little to save. Maker rescues the churny configs, raw persistence partial jumps 7779 to 9224, but those are the configs already rejected. For the chosen strategy the maker engine build is not clearly worth it.

## Deflated Sharpe

The Sharpe adjusted for the many configurations tried across the research. A high Sharpe from one of dozens of trials is less impressive than the same Sharpe from one. Deflated against an estimated trial count, with the return skew and kurtosis.

In [5]:
GAMMA = 0.5772156649


def deflated_sharpe(eq, n_trials):
    r = eq.set_index("ts")["equity"].resample("1D").last().ffill().diff().dropna()
    T = len(r); sr = r.mean() / r.std()
    g3 = float(skew(r)); g4 = float(kurtosis(r, fisher=False))
    sr_var = (1 - g3 * sr + (g4 - 1) / 4 * sr ** 2) / (T - 1)
    z1 = norm.ppf(1 - 1.0 / n_trials); z2 = norm.ppf(1 - 1.0 / (n_trials * np.e))
    sr0 = np.sqrt(sr_var) * ((1 - GAMMA) * z1 + GAMMA * z2)
    dsr = norm.cdf((sr - sr0) * np.sqrt(T - 1) / np.sqrt(1 - g3 * sr + (g4 - 1) / 4 * sr ** 2))
    return sr * np.sqrt(365), sr0 * np.sqrt(365), dsr


N_TRIALS = 60  # rough count of shape, sweep, policy and source configs tried
rows = []
for (sname, src), eq in eqs.items():
    if src not in ("psm6",):
        continue
    sr, sr0, dsr = deflated_sharpe(eq, N_TRIALS)
    rows.append({"config": f"{src} {sname}", "ann_sharpe": round(sr, 2),
                 "null_max_sharpe": round(sr0, 2), "deflated_sharpe_prob": round(dsr, 3)})
display(pd.DataFrame(rows).set_index("config"))

,ann_sharpe,null_max_sharpe,deflated_sharpe_prob
config,,,
psm6 threshold,4.46,1.41,1.0
psm6 partial,3.99,1.26,1.0


Deflated against about 60 trials the winner's Sharpe stays significant, the deflated probability is near one, the annual Sharpe sits well above the expected maximum under the null. The number is an approximation, the trial variance is taken from the estimator not the cross-trial spread, but the margin is wide enough that the edge is not a multiple-testing artifact.

## Read

The edge survives. Smoothed persistence on slow partial adjustment is the robust winner, positive and Sharpe above 3 at two times real cost, where raw persistence collapses and gbm trails. The ewma's low turnover is the source of the robustness, not a forecast. Deflated for the trial count the Sharpe holds.

Maker is not the lever it looked like. For the chosen low-turnover strategy the optimistic maker ceiling is about 6 percent, and the real figure after adverse selection is less. Maker pays only for high-turnover shapes that are already rejected. So the C++ maker build is deprioritized, the return to C++ is the live path, the margin and liquidation check, and the reverse leg on testnet, not maker.

One correction on the record. Earlier notebooks ran on a stale binary that undercharged cost, real fees about 380 against the correct 665. The relative verdicts are reconfirmed here on the cost-aware build, smoothed persistence beats gbm beats raw persistence at the correct higher cost, but the absolute PnL and fee drag in those notebooks are light and should be read as directional.